In [ ]:
import pandas as pd
import json
from pathlib import Path

DATA_PATH = "../data/processed/cleaned_transactions.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["InvoiceDate"])

print("Dataset loaded")
print(df.shape)

Dataset loaded
(367077, 12)


In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
print("Missing values per column:\n", missing_values)

assert missing_values.sum() == 0, "There should be no missing values in the cleaned dataset."

Missing values per column:
Invoice         0
StockCode       0
Description     0
Quantity        0
InvoiceDate     0
Price           0
Customer ID     0
Country         0
TotalPrice      0
InvoiceYear     0
InvoiceMonth    0
InvoiceDay      0
dtype: int64


In [ ]:
# Check that all quantities are positive
assert (df["Quantity"] > 0).all(), "All quantities should be positive."
print("Quantity check passed.")

Invalid quantity rows: 0


In [ ]:
# Check that all unit prices are positive
assert (df["UnitPrice"] > 0).all(), "All unit prices should be positive."
print("UnitPrice check passed.")

Invalid price rows: 0


In [ ]:
# Check CustomerID dtype is integer
print("CustomerID dtype:", df["CustomerID"].dtype)
assert pd.api.types.is_integer_dtype(df["CustomerID"]), "CustomerID must be an integer dtype."
print("CustomerID dtype check passed.")

float64


In [ ]:
# Check date range is valid (min < max and within expected bounds)
date_min, date_max = df["InvoiceDate"].min(), df["InvoiceDate"].max()
print("Date range:", date_min, "to", date_max)

assert date_min < date_max, "InvoiceDate min must be before max."
print("Date range check passed.")

CustomerID values are integer: True


In [ ]:
# Build standardized validation_report.json
from pathlib import Path

total_rows = int(len(df))
total_columns = int(len(df.columns))

date_min, date_max = df["InvoiceDate"].min(), df["InvoiceDate"].max()

validation_checks = {
    "no_missing_values": bool(df.isnull().sum().sum() == 0),
    "positive_quantities": bool((df["Quantity"] > 0).all()),
    "positive_unit_prices": bool((df["UnitPrice"] > 0).all()),
    "customerid_integer": bool(pd.api.types.is_integer_dtype(df["CustomerID"])),
    "valid_date_range": bool(date_min < date_max),
}

validation_report = {
    "total_rows": total_rows,
    "total_columns": total_columns,
    "date_range": {
        "start": str(date_min.date()),
        "end": str(date_max.date()),
    },
    "unique_customers": int(df["CustomerID"].nunique()),
    "unique_products": int(df["StockCode"].nunique()),
    "unique_countries": int(df["Country"].nunique()),
    "total_revenue": float(df["TotalPrice"].sum()),
    "average_order_value": float(df["TotalPrice"].mean()),
    "validation_passed": all(validation_checks.values()),
    "checks": validation_checks,
}

output_path = Path("../data/processed/validation_report.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w") as f:
    json.dump(validation_report, f, indent=4, default=str)

print("validation_report.json written to", output_path)

Validation report created.


: 